<a href="https://colab.research.google.com/github/kannanrk28/Capstone_Masai_FinalProject_Kannan/blob/main/part4/Part4_LLM_Powered_Feature.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ====================================================================================================
# Part 4 - Task 1 - Set up the LLM API connection
# ====================================================================================================


import os
import requests
#Assigning ENVIRONMENT KEY
try:
    from google.colab import userdata
    os.environ['LLM_API_KEY'] = userdata.get('LLM_API_KEY')
except Exception:
    pass
# #defining reusable LLM calling function
# def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):
#     api_key = os.environ.get('LLM_API_KEY')
#     if not api_key:
#         return None

#     url = "https://openrouter.ai/api/v1/chat/completions"

#     payload = {
#         "model": "openai/gpt-3.5-turbo",
#         "messages": [
#             {"role": "system", "content": system_prompt},
#             {"role": "user", "content": user_prompt}
#         ],
#         "temperature": temperature,
#         "max_tokens": max_tokens
#     }

#     headers = {
#         "Authorization": f"Bearer {api_key}",
#         "Content-Type": "application/json"
#     }

#     try:
#         response = requests.post(url, headers=headers, json=payload)
#         if response.status_code != 200:
#             print(f" API Error Status Code: {response.status_code}")
#             return None
#         return response.json()['choices'][0]['message']['content']
#     except Exception:
#         return None


# #Testing the prompt lively
# output = call_llm("You are a helpful assistant.", "Reply with only the word: hello")
# print(output)


In [ ]:
# ====================================================================================================
# 2. Reusable LLM API function
# ====================================================================================================
api_key = os.environ.get('LLM_API_KEY')
api_url = "https://openrouter.ai/api/v1/chat/completions"
def call_llm(
    system_prompt,
    user_prompt,
    temperature=0.0,
    max_tokens=512
):
    """
    Send a request to the LLM API and return only the assistant response text.
    """

    payload = {
        "model": "openai/gpt-3.5-turbo",
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    response = requests.post(
        api_url,
        headers=headers,
        json=payload,
        timeout=60
    )

    if response.status_code != 200:
        print("LLM request failed.")
        print("Status Code:", response.status_code)
        print("Response:", response.text)
        return None

    return response.json()["choices"][0]["message"]["content"]

#Testing the prompt lively
output = call_llm("You are a helpful assistant.", "Reply with only the word: hello")
print(output)

In [ ]:


# ====================================================================================================
# Prompt Designing - Zero Shot
# ====================================================================================================
import joblib
import pandas as pd
import json
import requests # Needed for downloading the model



# ====================================================================================================
# 3. Load the saved best pipeline
# ====================================================================================================

MODEL_RAW_URL = "https://raw.githubusercontent.com/kannanrk28/Capstone_Masai_FinalProject_Kannan/197f93a5a8e7e489351ca72f72f1746fc26d40d6/part4/best_model.pkl"
MODEL_LOCAL_PATH = "best_model.pkl"

try:
    response = requests.get(MODEL_RAW_URL)
    response.raise_for_status()  # Raise an exception for HTTP errors
    with open(MODEL_LOCAL_PATH, "wb") as f:
        f.write(response.content)
    best_model = joblib.load(MODEL_LOCAL_PATH)
    print("Best model downloaded and loaded successfully.")
except requests.exceptions.RequestException as e:
    print(f"Error downloading model: {e}")
    # Fallback or exit if download fails
    best_model = None # Ensure best_model is defined even if download fails
except Exception as e:
    print(f"Error loading model: {e}")
    best_model = None # Ensure best_model is defined even if loading fails

print("Model type:", type(best_model))


# ====================================================================================================
# 4. Helper function to encode one feature dictionary
# ====================================================================================================

def encode_record(features):
    """
    Convert one feature dictionary into a one-row DataFrame.

    The function also aligns the columns with the feature names
    used during model training.
    """

    record = pd.DataFrame([features])

    if hasattr(best_model, "feature_names_in_"):
        record = record.reindex(
            columns=best_model.feature_names_in_,
            fill_value=0
        )

    return record


# ====================================================================================================
# 5. Zero-shot system prompt
# ====================================================================================================

system_prompt = """
You are an AI assistant that explains machine learning classification predictions.

Your task is to generate a structured explanation for a movie revenue prediction.

You will receive:
1. A set of movie feature values.
2. The predicted class produced by the machine learning model.
3. The predicted probability for that class.

Return ONLY valid JSON using the following schema:

{
  "prediction_label": "string",
  "confidence_level": "low | medium | high",
  "top_reason": "string",
  "second_reason": "string",
  "next_step": "string"
}

Instructions:
- Return only valid JSON without Markdown or additional text.
- Do not invent or modify feature values.
- Do not claim that a feature caused the prediction.
- Describe only features that may have influenced the prediction.
- Base the explanation only on the supplied feature values and prediction.
- Determine confidence_level using the predicted probability:
    - high: probability >= 0.80
    - medium: probability >= 0.60 and < 0.80
    - low: probability < 0.60
- Keep each explanation concise and easy to understand.
"""


# ====================================================================================================
# 6. Create one hand-crafted feature-vector input
# ====================================================================================================

feature_values = {
    "budget": 150_000_000,
    "runtime": 130,
    "release_year": 2020,
    "release_month": 7,
    "Action": 1,
    "Adventure": 1,
    "Animation": 0,
    "Comedy": 0,
    "Crime": 0,
    "Documentary": 0,
    "Drama": 0,
    "Family": 0,
    "Fantasy": 1,
    "Foreign": 0,
    "History": 0,
    "Horror": 0,
    "Music": 0,
    "Mystery": 0,
    "Romance": 0,
    "Science Fiction": 1,
    "TV Movie": 0,
    "Thriller": 1,
    "War": 0,
    "Western": 0
}


# ====================================================================================================
# 7. Encode the input
# ====================================================================================================

# Check if best_model was loaded successfully before calling encode_record
if best_model is not None:
    encoded_record = encode_record(feature_values)

    print("\nEncoded Record Shape:")
    print(encoded_record.shape)

    print("\nEncoded Record:")
    print(encoded_record)
else:
    print("\nModel not loaded, skipping encoding.")
    encoded_record = None # Ensure encoded_record is defined


# ====================================================================================================
# 8. Generate model prediction
# ====================================================================================================

predicted_class = None
prediction_label = None
predicted_probability = None

if best_model is not None and encoded_record is not None:
    predicted_class = int(
        best_model.predict(encoded_record)[0]
    )

    class_probabilities = best_model.predict_proba(encoded_record)[0]

    class_0_probability = float(class_probabilities[0])
    class_1_probability = float(class_probabilities[1])

    # Probability associated with the class that was actually predicted
    predicted_probability = (
        class_1_probability
        if predicted_class == 1
        else class_0_probability
    )

    prediction_label = (
        "High Revenue"
        if predicted_class == 1
        else "Low Revenue"
    )

    print("\nPredicted Class:", predicted_class)
    print("Prediction Label:", prediction_label)
    print(f"Class 0 Probability: {class_0_probability:.6f}")
    print(f"Class 1 Probability: {class_1_probability:.6f}")
    print(f"Predicted Class Probability: {predicted_probability:.6f}")
else:
    print("\nModel or encoded record not available, skipping prediction.")


# ====================================================================================================
# 9. Construct the user prompt
# ====================================================================================================

user_prompt = None
if predicted_class is not None and prediction_label is not None and predicted_probability is not None:
    user_prompt = f"""
Movie Features:
{json.dumps(feature_values, indent=2)}

Predicted Class:
{predicted_class}

Prediction Label:
{prediction_label}

Predicted Probability:
{predicted_probability:.6f}

Generate the structured JSON explanation.
""".strip()

    print("\nUser Prompt:")
    print(user_prompt)
else:
    print("\nSkipping user prompt construction due to missing prediction data.")


# ====================================================================================================
# 10. Call the LLM using temperature = 0
# ====================================================================================================

raw_llm_response = None
if user_prompt is not None:
    raw_llm_response = call_llm(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        temperature=0.0,
        max_tokens=512
    )

    print("\nRaw LLM Response:")
    print(raw_llm_response)
else:
    print("\nSkipping LLM call due to missing user prompt.")


# ====================================================================================================
# 11. Parse the JSON response
# ====================================================================================================

if raw_llm_response is not None:
    try:
        parsed_response = json.loads(raw_llm_response.strip())

        print("\nParsed JSON Response:")
        print(json.dumps(parsed_response, indent=2))

    except json.JSONDecodeError as error:
        print("\nThe LLM response is not valid JSON.")
        print("JSON Error:", error)
else:
    print("\nNo LLM response was returned.")

In [ ]:
# ====================================================================================================
# Prompt Designing - Temperature Comparison
# ====================================================================================================


import json
import pandas as pd

# Assumes these already exist:
# best_model
# encode_record()
# build_user_prompt()
# call_llm()
# system_prompt
# test_inputs

import json

def build_user_prompt(
    feature_values,
    predicted_class,
    predicted_probability
):
    """
    Build the user prompt for the LLM.
    """

    prediction_label = (
        "High Revenue"
        if predicted_class == 1
        else "Low Revenue"
    )

    prompt = f"""
Movie Features:

{json.dumps(feature_values, indent=2)}

Predicted Class:
{predicted_class}

Prediction Label:
{prediction_label}

Predicted Probability:
{predicted_probability:.4f}

Generate a structured JSON explanation using this schema:

{{
  "prediction_label": "string",
  "confidence_level": "low | medium | high",
  "top_reason": "string",
  "second_reason": "string",
  "next_step": "string"
}}

Return ONLY valid JSON.
"""

    return prompt
import re

def has_pii(text):
    """
    Detect email addresses or phone numbers
    in the input text.
    """

    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'

    phone_pattern = (
        r'\b\d{10}\b|'
        r'\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    )

    return bool(
        re.search(email_pattern, text) or
        re.search(phone_pattern, text)
    )

temperature_results = []
# Three hand-crafted feature-vector inputs

test_inputs = [

    # -------------------------------
    # Movie 1
    # -------------------------------
    {
        "budget": 150000000,
        "runtime": 130,
        "release_year": 2020,
        "release_month": 7,
        "Action": 1,
        "Adventure": 1,
        "Animation": 0,
        "Comedy": 0,
        "Crime": 0,
        "Documentary": 0,
        "Drama": 0,
        "Family": 0,
        "Fantasy": 1,
        "Foreign": 0,
        "History": 0,
        "Horror": 0,
        "Music": 0,
        "Mystery": 0,
        "Romance": 0,
        "Science Fiction": 1,
        "TV Movie": 0,
        "Thriller": 1,
        "War": 0,
        "Western": 0
    },

    # -------------------------------
    # Movie 2
    # -------------------------------
    {
        "budget": 10000000,
        "runtime": 95,
        "release_year": 2018,
        "release_month": 2,
        "Action": 0,
        "Adventure": 0,
        "Animation": 0,
        "Comedy": 1,
        "Crime": 0,
        "Documentary": 0,
        "Drama": 1,
        "Family": 0,
        "Fantasy": 0,
        "Foreign": 0,
        "History": 0,
        "Horror": 0,
        "Music": 0,
        "Mystery": 0,
        "Romance": 1,
        "Science Fiction": 0,
        "TV Movie": 0,
        "Thriller": 0,
        "War": 0,
        "Western": 0
    },

    # -------------------------------
    # Movie 3
    # -------------------------------
    {
        "budget": 45000000,
        "runtime": 110,
        "release_year": 2015,
        "release_month": 10,
        "Action": 0,
        "Adventure": 0,
        "Animation": 0,
        "Comedy": 0,
        "Crime": 1,
        "Documentary": 0,
        "Drama": 1,
        "Family": 0,
        "Fantasy": 0,
        "Foreign": 0,
        "History": 0,
        "Horror": 0,
        "Music": 0,
        "Mystery": 1,
        "Romance": 0,
        "Science Fiction": 0,
        "TV Movie": 0,
        "Thriller": 1,
        "War": 0,
        "Western": 0
    }

]

for input_number, feature_values in enumerate(test_inputs, start=1):

    # Prepare the feature row for the saved pipeline
    encoded_record = encode_record(feature_values)

    # Generate ML prediction
    predicted_class = int(
        best_model.predict(encoded_record)[0]
    )

    class_probabilities = best_model.predict_proba(encoded_record)[0]

    class_1_probability = float(class_probabilities[1])

    predicted_probability = (
        class_1_probability
        if predicted_class == 1
        else 1 - class_1_probability
    )

    # Build the same prompt for both temperature settings
    user_prompt = build_user_prompt(
        feature_values=feature_values,
        predicted_class=predicted_class,
        predicted_probability=predicted_probability
    )

    # PII guardrail before the LLM calls
    if has_pii(user_prompt):
        print(f"Input {input_number} blocked: PII detected.")
        continue

    # Call 1: deterministic setting
    output_temp_0 = call_llm(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        temperature=0.0,
        max_tokens=512
    )

    # Call 2: higher-variability setting
    output_temp_07 = call_llm(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        temperature=0.7,
        max_tokens=512
    )

    temperature_results.append({
        "Input": f"Input {input_number}",
        "Feature Values": feature_values,
        "Predicted Class": predicted_class,
        "Predicted Probability": round(predicted_probability, 6),
        "Output at temp=0": output_temp_0,
        "Output at temp=0.7": output_temp_07
    })

    print("\n" + "=" * 80)
    print(f"INPUT {input_number}")
    print("=" * 80)

    print("\nFeature Values:")
    print(json.dumps(feature_values, indent=2))

    print("\nPredicted Class:")
    print(predicted_class)

    print("\nPredicted Probability:")
    print(f"{predicted_probability:.6f}")

    print("\nOutput at temperature = 0:")
    print(output_temp_0)

    print("\nOutput at temperature = 0.7:")
    print(output_temp_07)


# Compact comparison table
temperature_table = pd.DataFrame([
    {
        "Input": item["Input"],
        "Output at temp=0": item["Output at temp=0"],
        "Output at temp=0.7": item["Output at temp=0.7"]
    }
    for item in temperature_results
])

print("\nTemperature Comparison Table")
print(temperature_table)

In [ ]:
# ====================================================================================================
# Structure Output Validation
# ====================================================================================================

import json
import pandas as pd
import joblib
import requests # Import requests for downloading the model

from jsonschema import validate
from jsonschema.exceptions import ValidationError

# ==========================================================
# Load best model
# ==========================================================

MODEL_RAW_URL = "https://raw.githubusercontent.com/kannanrk28/Capstone_Masai_FinalProject_Kannan/197f93a5a8e7e489351ca72f72f1746fc26d40d6/part4/best_model.pkl"
MODEL_LOCAL_PATH = "best_model_validation.pkl" # Use a different local name to avoid conflicts

try:
    response = requests.get(MODEL_RAW_URL)
    response.raise_for_status()  # Raise an exception for HTTP errors
    with open(MODEL_LOCAL_PATH, "wb") as f:
        f.write(response.content)
    best_model = joblib.load(MODEL_LOCAL_PATH)
    print("Best model downloaded and loaded successfully for validation.")
except requests.exceptions.RequestException as e:
    print(f"Error downloading model for validation: {e}")
    best_model = None
except Exception as e:
    print(f"Error loading model for validation: {e}")
    best_model = None

# ==========================================================
# Define the required JSON schema
# ==========================================================

explanation_schema = {
    "type": "object",
    "properties": {
        "prediction_label": {
            "type": ["string", "null"]
        },
        "confidence_level": {
            "type": ["string", "null"],
            "enum": ["low", "medium", "high", None]
        },
        "top_reason": {
            "type": ["string", "null"]
        },
        "second_reason": {
            "type": ["string", "null"]
        },
        "next_step": {
            "type": ["string", "null"]
        }
    },
    "required": [
        "prediction_label",
        "confidence_level",
        "top_reason",
        "second_reason",
        "next_step"
    ],
    "additionalProperties": False
}

# ==========================================================
# Fallback JSON
# ==========================================================

fallback = {
    "prediction_label": None,
    "confidence_level": None,
    "top_reason": None,
    "second_reason": None,
    "next_step": None
}

# ==========================================================
# Three hand-crafted inputs
# ==========================================================

test_inputs = [
    {
        "budget":150000000,
        "runtime":130,
        "release_year":2020,
        "release_month":7,
        "Action":1,
        "Adventure":1,
        "Animation":0,
        "Comedy":0,
        "Crime":0,
        "Documentary":0,
        "Drama":0,
        "Family":0,
        "Fantasy":1,
        "Foreign":0,
        "History":0,
        "Horror":0,
        "Music":0,
        "Mystery":0,
        "Romance":0,
        "Science Fiction":1,
        "TV Movie":0,
        "Thriller":1,
        "War":0,
        "Western":0
    },

    {
        "budget":10000000,
        "runtime":95,
        "release_year":2018,
        "release_month":2,
        "Action":0,
        "Adventure":0,
        "Animation":0,
        "Comedy":1,
        "Crime":0,
        "Documentary":0,
        "Drama":1,
        "Family":0,
        "Fantasy":0,
        "Foreign":0,
        "History":0,
        "Horror":0,
        "Music":0,
        "Mystery":0,
        "Romance":1,
        "Science Fiction":0,
        "TV Movie":0,
        "Thriller":0,
        "War":0,
        "Western":0
    },

    {
        "budget":45000000,
        "runtime":110,
        "release_year":2015,
        "release_month":10,
        "Action":0,
        "Adventure":0,
        "Animation":0,
        "Comedy":0,
        "Crime":1,
        "Documentary":0,
        "Drama":1,
        "Family":0,
        "Fantasy":0,
        "Foreign":0,
        "History":0,
        "Horror":0,
        "Music":0,
        "Mystery":1,
        "Romance":0,
        "Science Fiction":0,
        "TV Movie":0,
        "Thriller":1,
        "War":0,
        "Western":0
    }
]

results = []

# Ensure best_model was loaded before proceeding
if best_model is not None:
    for idx, features in enumerate(test_inputs, start=1):

        # ---------------------------------------------
        # Encode
        # ---------------------------------------------
        X = encode_record(features)

        # ---------------------------------------------
        # Prediction
        # ---------------------------------------------
        pred = int(best_model.predict(X)[0])

        prob = float(best_model.predict_proba(X)[0][pred])

        # ---------------------------------------------
        # Prompt
        # ---------------------------------------------
        user_prompt = build_user_prompt(
            features,
            pred,
            prob
        )

        # ---------------------------------------------
        # Guardrail
        # ---------------------------------------------
        if has_pii(user_prompt):

            print("Input blocked: PII detected.")

            explanation = fallback

            status = "Blocked"

        else:

            raw_response = call_llm(
                system_prompt,
                user_prompt,
                temperature=0
            )

            try:

                explanation = json.loads(raw_response.strip())

                validate(
                    instance=explanation,
                    schema=explanation_schema
                )

                status = "Passed"

            except json.JSONDecodeError as e:

                print("JSON Error:", e)

                explanation = fallback

                status = "JSON Failed"

            except ValidationError as e:

                print("Schema Error:", e.message)

                explanation = fallback

                status = "Schema Failed"

        # ---------------------------------------------
        # Display
        # ---------------------------------------------
        print("="*80)

        print("Input", idx)

        print("\nFeature Values")
        print(features)

        print("\nPredicted Class")
        print(pred)

        print("\nProbability")
        print(prob)

        print("\nExplanation")
        print(explanation)

        print("\nValidation")
        print(status)

        results.append({
            "Feature Input": f"Input {idx}",
            "Predicted Class": pred,
            "Probability": round(prob,4),
            "Explanation JSON": explanation,
            "Validation Status": status
        })

    results_df = pd.DataFrame(results)

    print("\nValidation Results DataFrame:")
    print(results_df)
else:
    print("\nSkipping validation due to model loading failure.")

In [ ]:

# ====================================================================================================
# Part 4 - Guardrails - PII Detection
# ====================================================================================================

import re

def has_pii(text):
    """
    Detect email addresses and phone numbers.
    """

    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'

    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'

    return bool(
        re.search(email_pattern, text) or
        re.search(phone_pattern, text)
    )
# ====================================================================================================
# Safe LLM Call
# ====================================================================================================

def safe_call_llm(
    system_prompt,
    user_input,
    temperature=0.0,
    max_tokens=512
):
    """
    Apply PII guardrail before calling the LLM.
    """

    if has_pii(user_input):
        print("Input blocked: PII detected.")
        return None

    return call_llm(
        system_prompt,
        user_input,
        temperature,
        max_tokens
    )
# ====================================================================================================
# Test Case 1 - Contains Email
# ====================================================================================================

user_input = """
Customer email is john.doe@gmail.com.

Please explain the prediction.
"""

response = safe_call_llm(
    system_prompt,
    user_input,
    temperature=0
)

print(response)

# ====================================================================================================
# Test Case 2 - No PII
# ====================================================================================================

user_input = """
Movie Budget : 150000000

Runtime : 130

Predicted Class : High Revenue

Predicted Probability : 0.95

Explain the prediction.
"""

response = safe_call_llm(
    system_prompt,
    user_input,
    temperature=0
)

print(response)

In [ ]:
# ====================================================================================================
# Part 4 - End-to-End Demonstration for Track C
# ====================================================================================================

import json
import re
import joblib
import pandas as pd

from jsonschema import validate
from jsonschema.exceptions import ValidationError


# ----------------------------------------------------------------------------------------------------
# 1. Load the saved best-performing pipeline
# ----------------------------------------------------------------------------------------------------

best_model = joblib.load("best_model.pkl")


# ----------------------------------------------------------------------------------------------------
# 2. Required JSON schema
# ----------------------------------------------------------------------------------------------------

explanation_schema = {
    "type": "object",
    "properties": {
        "prediction_label": {
            "type": "string"
        },
        "confidence_level": {
            "type": "string",
            "enum": ["low", "medium", "high"]
        },
        "top_reason": {
            "type": "string"
        },
        "second_reason": {
            "type": "string"
        },
        "next_step": {
            "type": "string"
        }
    },
    "required": [
        "prediction_label",
        "confidence_level",
        "top_reason",
        "second_reason",
        "next_step"
    ],
    "additionalProperties": False
}


# ----------------------------------------------------------------------------------------------------
# 3. Fallback response
# ----------------------------------------------------------------------------------------------------

fallback_explanation = {
    "prediction_label": None,
    "confidence_level": None,
    "top_reason": None,
    "second_reason": None,
    "next_step": None
}


# ----------------------------------------------------------------------------------------------------
# 4. PII guardrail
# ----------------------------------------------------------------------------------------------------

def has_pii(text):
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'

    return bool(
        re.search(email_pattern, text) or
        re.search(phone_pattern, text)
    )


# ----------------------------------------------------------------------------------------------------
# 5. Encode one feature record
# ----------------------------------------------------------------------------------------------------

def encode_record(features):
    record = pd.DataFrame([features])

    if hasattr(best_model, "feature_names_in_"):
        record = record.reindex(
            columns=best_model.feature_names_in_,
            fill_value=0
        )

    return record


# ----------------------------------------------------------------------------------------------------
# 6. Build the user prompt
# ----------------------------------------------------------------------------------------------------

def build_user_prompt(
    feature_values,
    predicted_class,
    predicted_probability
):
    prediction_label = (
        "High Revenue"
        if predicted_class == 1
        else "Low Revenue"
    )

    return f"""
Movie Features:
{json.dumps(feature_values, indent=2)}

Predicted Class:
{predicted_class}

Prediction Label:
{prediction_label}

Predicted Probability:
{predicted_probability:.6f}

Return only valid JSON using exactly this schema:

{{
  "prediction_label": "string",
  "confidence_level": "low | medium | high",
  "top_reason": "string",
  "second_reason": "string",
  "next_step": "string"
}}

Rules:
- Return JSON only.
- Do not include Markdown or extra text.
- Do not add extra fields.
- Do not invent feature values.
- Do not claim that a feature caused the prediction.
- Use cautious wording such as "may have influenced".
""".strip()


# ----------------------------------------------------------------------------------------------------
# 7. Parse and validate LLM output
# ----------------------------------------------------------------------------------------------------

def parse_and_validate(raw_response):
    if raw_response is None:
        print("Validation error: no LLM response was returned.")
        return fallback_explanation.copy(), "Fail"

    cleaned_response = raw_response.strip()

    if cleaned_response.startswith("```json"):
        cleaned_response = cleaned_response[7:]
    elif cleaned_response.startswith("```"):
        cleaned_response = cleaned_response[3:]

    if cleaned_response.endswith("```"):
        cleaned_response = cleaned_response[:-3]

    cleaned_response = cleaned_response.strip()

    try:
        parsed_response = json.loads(cleaned_response)

    except json.JSONDecodeError as error:
        print("JSON parsing error:", error)
        return fallback_explanation.copy(), "Fail"

    try:
        validate(
            instance=parsed_response,
            schema=explanation_schema
        )

    except ValidationError as error:
        print("Schema validation error:", error.message)
        return fallback_explanation.copy(), "Fail"

    return parsed_response, "Pass"


# ----------------------------------------------------------------------------------------------------
# 8. Three distinct feature-vector inputs
# ----------------------------------------------------------------------------------------------------

test_inputs = [
    {
        "budget": 150_000_000,
        "runtime": 130,
        "release_year": 2020,
        "release_month": 7,
        "Action": 1,
        "Adventure": 1,
        "Animation": 0,
        "Comedy": 0,
        "Crime": 0,
        "Documentary": 0,
        "Drama": 0,
        "Family": 0,
        "Fantasy": 1,
        "Foreign": 0,
        "History": 0,
        "Horror": 0,
        "Music": 0,
        "Mystery": 0,
        "Romance": 0,
        "Science Fiction": 1,
        "TV Movie": 0,
        "Thriller": 1,
        "War": 0,
        "Western": 0
    },
    {
        "budget": 10_000_000,
        "runtime": 95,
        "release_year": 2018,
        "release_month": 2,
        "Action": 0,
        "Adventure": 0,
        "Animation": 0,
        "Comedy": 1,
        "Crime": 0,
        "Documentary": 0,
        "Drama": 1,
        "Family": 0,
        "Fantasy": 0,
        "Foreign": 0,
        "History": 0,
        "Horror": 0,
        "Music": 0,
        "Mystery": 0,
        "Romance": 1,
        "Science Fiction": 0,
        "TV Movie": 0,
        "Thriller": 0,
        "War": 0,
        "Western": 0
    },
    {
        "budget": 45_000_000,
        "runtime": 110,
        "release_year": 2015,
        "release_month": 10,
        "Action": 0,
        "Adventure": 0,
        "Animation": 0,
        "Comedy": 0,
        "Crime": 1,
        "Documentary": 0,
        "Drama": 1,
        "Family": 0,
        "Fantasy": 0,
        "Foreign": 0,
        "History": 0,
        "Horror": 0,
        "Music": 0,
        "Mystery": 1,
        "Romance": 0,
        "Science Fiction": 0,
        "TV Movie": 0,
        "Thriller": 1,
        "War": 0,
        "Western": 0
    }
]


# ----------------------------------------------------------------------------------------------------
# 9. Run the complete pipeline
# ----------------------------------------------------------------------------------------------------

results = []

for input_number, feature_values in enumerate(test_inputs, start=1):

    print("\n" + "=" * 90)
    print(f"INPUT {input_number}")
    print("=" * 90)

    print("\nFeature Input:")
    print(json.dumps(feature_values, indent=2))

    encoded_record = encode_record(feature_values)

    predicted_class = int(
        best_model.predict(encoded_record)[0]
    )

    class_probabilities = best_model.predict_proba(encoded_record)[0]

    predicted_probability = float(
        class_probabilities[predicted_class]
    )

    user_prompt = build_user_prompt(
        feature_values=feature_values,
        predicted_class=predicted_class,
        predicted_probability=predicted_probability
    )

    # Guardrail before every LLM call
    if has_pii(user_prompt):
        print("\nInput blocked: PII detected.")

        raw_response = None
        guardrail_result = "Block"
        validated_explanation = fallback_explanation.copy()
        validation_outcome = "Fail"

    else:
        guardrail_result = "Pass"

        raw_response = call_llm(
            system_prompt=system_prompt,
            user_prompt=user_prompt,
            temperature=0.0,
            max_tokens=512
        )

        validated_explanation, validation_outcome = parse_and_validate(
            raw_response
        )

    print("\nPredicted Class:")
    print(predicted_class)

    print("\nPredicted Probability:")
    print(f"{predicted_probability:.6f}")

    print("\nLLM Raw Response:")
    print(raw_response)

    print("\nValidation Outcome:")
    print(validation_outcome)

    print("\nGuardrail Result:")
    print(guardrail_result)

    print("\nValidated Explanation:")
    print(json.dumps(validated_explanation, indent=2))

    results.append({
        "Input": f"Input {input_number}",
        "LLM Output": raw_response,
        "Valid JSON": validation_outcome,
        "Pass/Block": guardrail_result
    })


# ----------------------------------------------------------------------------------------------------
# 10. Display summary table
# ----------------------------------------------------------------------------------------------------

results_table = pd.DataFrame(results)

print("\n" + "=" * 90)
print("END-TO-END SUMMARY")
print("=" * 90)

print(results_table)